Search for matching candidates given a job profile.

Requires both Elasticsearch indices:
- `job_profiles` (job embeddings)
- `candidate_profiles` (candidate embeddings)

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from core.config import EmbeddingConfig, ElasticsearchConfig
from core.elasticsearch import get_es_client
from core.indices import CANDIDATE_INDEX, JOB_INDEX
from core.search import (
    search_candidates_by_job_id,
    search_candidates_by_profile_text,
)

K = 10
NUM_CANDIDATES = 100
BATCH_JOBS = 10  # for optional batch export
OUTPUT_PATH = Path('../data/job_candidate_matches.jsonl')

embed_config = EmbeddingConfig.from_env()
es = get_es_client(ElasticsearchConfig.from_env())

In [ ]:
for index in (JOB_INDEX, CANDIDATE_INDEX):
    count = es.count(index=index)['count']
    print(f'{index}: {count:,} documents')

In [ ]:
# Pick a sample job id from the index
sample = es.search(index=JOB_INDEX, size=1, _source=['id', 'position', 'primary_keyword'])
job_hit = sample['hits']['hits'][0]
job_id = job_hit['_id']
job_source = job_hit['_source']

print(f"Sample job id: {job_id}")
print(f"Position: {job_source.get('position', '')!r}")
print(f"Keyword: {job_source.get('primary_keyword', '')!r}")

In [ ]:
# Search by stored job embedding (no API call)
hits = search_candidates_by_job_id(
    es,
    job_id,
    k=K,
    num_candidates=NUM_CANDIDATES,
)

print(f'Top {K} candidates for job {job_id}:')
for hit in hits:
    print(
        f"  score={hit['score']:.4f}  "
        f"id={hit['id']}  "
        f"position={hit.get('position', '')!r}  "
        f"keyword={hit.get('primary_keyword', '')!r}"
    )

In [ ]:
# Search by job profile_text (embed at query time)
job_doc = es.get(index=JOB_INDEX, id=job_id)
profile_text = job_doc['_source']['profile_text']

hits_live = search_candidates_by_profile_text(
    profile_text,
    es,
    k=K,
    num_candidates=NUM_CANDIDATES,
    config=embed_config,
)

print(f'Top {K} candidates via live embed:')
for hit in hits_live:
    print(
        f"  score={hit['score']:.4f}  "
        f"id={hit['id']}  "
        f"position={hit.get('position', '')!r}"
    )

In [ ]:
# Optional: batch top-k matches for first N jobs
job_ids = [
    hit['_id']
    for hit in es.search(index=JOB_INDEX, size=BATCH_JOBS, _source=False)['hits']['hits']
]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w') as f:
    for jid in job_ids:
        matches = search_candidates_by_job_id(
            es,
            jid,
            k=K,
            num_candidates=NUM_CANDIDATES,
        )
        record = {
            'job_id': jid,
            'candidates': [
                {'candidate_id': m['id'], 'score': m['score']} for m in matches
            ],
        }
        f.write(json.dumps(record) + '\n')

print(f'Wrote {len(job_ids)} job match records to {OUTPUT_PATH}')